**TRAIN MODEL CÁC TỈNH KHÁC**

Mount vào drive

In [1]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)
%cd '/content/gdrive/MyDrive/Python_Thu5/project'

Mounted at /content/gdrive
/content/gdrive/MyDrive/Python_Thu5/project


Import thư viện

In [2]:
import pandas as pd
import numpy as np
import joblib

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

Load dataset

In [3]:
df = pd.read_csv('housing_cleaned.csv')

Tiền xử lí

In [4]:
df.columns = [col.strip() for col in df.columns]

df['Tỉnh/Thành phố'] = df['Tỉnh/Thành phố'].replace({
    'Hà Nội.': 'Hà Nội',
    'Tp Hồ Chí Minh': 'Hồ Chí Minh',
    'TP Hồ Chí Minh': 'Hồ Chí Minh'
}).astype(str).str.strip()

def fill_geo(row):
    if pd.isnull(row['Quận']) or pd.isnull(row['Huyện']):
        parts = [p.strip() for p in str(row['Địa chỉ']).split(',')]
        if len(parts) >= 2:
            val = parts[-2]
            if pd.isnull(row['Quận']):
                row['Quận'] = val
            if pd.isnull(row['Huyện']):
                row['Huyện'] = val
    return row

df = df.apply(fill_geo, axis=1)

df['Loại hình nhà ở'] = df['Loại hình nhà ở'].fillna('Không xác định')
df['Giấy tờ pháp lý'] = df['Giấy tờ pháp lý'].fillna('Đang cập nhật')
df['Số tầng'] = df['Số tầng'].fillna(df['Số tầng'].median())
df['Số phòng ngủ'] = df['Số phòng ngủ'].fillna(df['Số phòng ngủ'].median())

city_counts = df['Tỉnh/Thành phố'].value_counts()
df = df[df['Tỉnh/Thành phố'].isin(city_counts[city_counts >= 300].index)]


# =============================
# LẤY CÁC TỈNH NGOÀI HÀ NỘI
# =============================
df = df[df['Tỉnh/Thành phố'] != 'Hà Nội'].copy()

features = [
    'Tỉnh/Thành phố', 'Quận', 'Loại hình nhà ở',
    'Số tầng', 'Số phòng ngủ', 'Diện tích', 'Giấy tờ pháp lý'
]
target = 'Giá (triệu đồng/m2)'

df = df[features + [target]].dropna(subset=[target])
df[target] = np.log1p(df[target])

cat_cols = ['Tỉnh/Thành phố', 'Quận', 'Loại hình nhà ở', 'Giấy tờ pháp lý']

for col in cat_cols:
    df[col] = df[col].fillna('Unknown')


target encoding

In [5]:
encoding_maps = {}
for col in cat_cols:
    m = df.groupby(col)[target].mean().to_dict()
    encoding_maps[col] = m
    df[col] = df[col].map(m)

joblib.dump(encoding_maps, 'encoding_maps_others.pkl')

['encoding_maps_others.pkl']

Tiến hành train

In [6]:
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

Đánh giá kết quả dùng R2 + MAE

In [7]:
print("R2:", r2_score(y_test, pred))
print("MAE:", mean_absolute_error(np.expm1(y_test), np.expm1(pred)))

R2: 0.7875938429158487
MAE: 16.428916136481412


Lưu model

In [8]:
joblib.dump(model, 'xgboost_other_provinces.pkl')
print("Saved xgboost_other_provinces.pkl")

Saved xgboost_other_provinces.pkl
